# LeetCode #12 — Integer to Roman: thuật toán tham lam (Greedy) với bảng giá trị

**Mức độ:** Medium · **Chủ đề:** Hash Table, Math, String, Greedy

Skeleton trên LeetCode:

```python
class Solution:
    def intToRoman(self, num: int) -> str:
```

Ở bài #13 Roman to Integer, bạn đã **đọc** số La Mã. Bài này làm chiều ngược lại: **viết** một số nguyên thành số La Mã.

Mục tiêu của notebook là giúp bạn hiểu **kỹ thuật tham lam (greedy)**: ở mỗi bước, lấy **ký hiệu lớn nhất còn dùng được**. Và quan trọng: hiểu **vì sao** đưa các cặp trừ (`CM`, `XL`, …) vào bảng thì greedy cho kết quả đúng.

---

## Sau notebook này, bạn sẽ nắm được

1. Quy tắc viết số La Mã, đặc biệt là 6 dạng trừ `IV, IX, XL, XC, CD, CM`.
2. Cách tách số thành từng hàng (nghìn, trăm, chục, đơn vị) bằng `//` và `%`.
3. Cách giải theo từng hàng với bảng tra.
4. Cách giải **greedy** với bảng 13 giá trị, và vì sao nó đúng.
5. Kiểm tra bằng cách **đi vòng**: số → La Mã → số.

# 1. Bài toán là gì?

Bảy ký hiệu La Mã:

| Ký hiệu | I | V | X | L | C | D | M |
|---|---|---|---|---|---|---|---|
| Giá trị | 1 | 5 | 10 | 50 | 100 | 500 | 1000 |

Quy tắc viết (theo đề):

1. Viết từ **giá trị lớn tới nhỏ**, từ trái sang phải, cộng dồn lại.
2. Nếu chữ số của một hàng **không bắt đầu bằng 4 hoặc 9**: chọn ký hiệu lớn nhất có thể trừ ra, viết nó, rồi xử lý phần còn lại. Ví dụ `8 = V + III = VIII`.
3. Nếu chữ số **bắt đầu bằng 4 hoặc 9**: dùng **dạng trừ** — một ký hiệu nhỏ đứng **trước** ký hiệu lớn. Chỉ có 6 dạng: `IV (4)`, `IX (9)`, `XL (40)`, `XC (90)`, `CD (400)`, `CM (900)`.
4. `I, X, C, M` lặp **tối đa 3 lần** liên tiếp. `V, L, D` **không lặp**.

**Constraints:** `1 <= num <= 3999`

---

## Ví dụ

| num | Kết quả | Phân tích |
|---|---|---|
| 3749 | `MMMDCCXLIX` | 3000 = MMM, 700 = DCC, 40 = XL, 9 = IX |
| 58 | `LVIII` | 50 = L, 8 = VIII |
| 1994 | `MCMXCIV` | 1000 = M, 900 = CM, 90 = XC, 4 = IV |

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Thuật toán tham lam (greedy)
2. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [1]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [2]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [3]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [4]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 314.79 ms | trên set: 0.079 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [5]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [6]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Áp dụng vào bài này

Đổi sang số La Mã giống **đổi tiền tham lam**: luôn lấy "mệnh giá" lớn nhất không vượt số còn lại, với bảng 13 mệnh giá (gồm cả `CM`, `XC`, `IV`…). Hệ La Mã được thiết kế để tham lam luôn đúng. Bảng ký hiệu ↔ giá trị có thể lưu bằng `dict` hoặc list các cặp.

# 3. Nền tảng: tách số theo hàng

Với `num = 3749`:

```text
nghìn  = num // 1000        = 3
trăm   = (num // 100) % 10  = 7
chục   = (num // 10) % 10   = 4
đơn vị = num % 10           = 9
```

`num // 100` bỏ hai chữ số cuối (`37`), `% 10` lấy chữ số cuối của phần còn lại (`7`).

In [7]:
num = 3749
thousands = num // 1000
hundreds = (num // 100) % 10
tens = (num // 10) % 10
ones = num % 10
print(thousands, hundreds, tens, ones)
print(thousands * 1000 + hundreds * 100 + tens * 10 + ones == num)

3 7 4 9
True


# 4. Cách 1: mỗi hàng viết độc lập bằng bảng tra

Quan sát: cách viết mỗi chữ số `0..9` ở **mọi hàng** có cùng "hình dạng", chỉ khác bộ ký hiệu:

| Chữ số | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| Đơn vị (I, V, X) | | I | II | III | IV | V | VI | VII | VIII | IX |
| Chục (X, L, C) | | X | XX | XXX | XL | L | LX | LXX | LXXX | XC |
| Trăm (C, D, M) | | C | CC | CCC | CD | D | DC | DCC | DCCC | CM |
| Nghìn (M) | | M | MM | MMM | | | | | | |

(Hàng nghìn chỉ tới 3 vì `num <= 3999`.)

Vậy ta chỉ cần **tra bảng** cho từng hàng rồi ghép lại.

In [8]:
ONES = ["", "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX"]
TENS = ["", "X", "XX", "XXX", "XL", "L", "LX", "LXX", "LXXX", "XC"]
HUNDREDS = ["", "C", "CC", "CCC", "CD", "D", "DC", "DCC", "DCCC", "CM"]
THOUSANDS = ["", "M", "MM", "MMM"]

def int_to_roman_table(num: int) -> str:
    return (THOUSANDS[num // 1000]
            + HUNDREDS[(num // 100) % 10]
            + TENS[(num // 10) % 10]
            + ONES[num % 10])

for n in [3749, 58, 1994, 4, 3999]:
    print(n, "->", int_to_roman_table(n))

3749 -> MMMDCCXLIX
58 -> LVIII
1994 -> MCMXCIV
4 -> IV
3999 -> MMMCMXCIX


## Ưu và nhược

- **Ưu:** rất nhanh (4 lần tra bảng), không thể sai nếu bảng đúng.
- **Nhược:** phải gõ tay 33 chuỗi — dễ gõ nhầm; và không cho ta thấy **quy luật** chung.

Ta hãy tìm một cách "tự sinh" ra các chuỗi này.

# 5. Cách 2: Greedy — lấy ký hiệu lớn nhất có thể

Ý tưởng giống **thối tiền**: muốn trả `58` đồng bằng ít tờ nhất, ta lấy tờ lớn nhất không vượt quá số tiền còn lại, lặp lại.

```text
58: lấy L (50) -> còn 8
 8: lấy V (5)  -> còn 3
 3: lấy I      -> còn 2 -> lấy I -> còn 1 -> lấy I -> còn 0
=> LVIII
```

## Nhưng chỉ 7 ký hiệu là chưa đủ

Với `4`: greedy chỉ với 7 ký hiệu sẽ cho `IIII` (lấy I bốn lần) — **sai luật** (I không lặp quá 3 lần, phải là `IV`).

**Mẹo:** coi 6 dạng trừ như **6 "tờ tiền" mới**. Bảng đầy đủ có 13 giá trị, xếp giảm dần:

| M | CM | D | CD | C | XC | L | XL | X | IX | V | IV | I |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 1000 | 900 | 500 | 400 | 100 | 90 | 50 | 40 | 10 | 9 | 5 | 4 | 1 |

Bây giờ với `4`: tờ lớn nhất ≤ 4 là `IV` → đúng ngay.

In [9]:
def greedy_seven_symbols(num: int) -> str:
    values = [(1000, "M"), (500, "D"), (100, "C"), (50, "L"), (10, "X"), (5, "V"), (1, "I")]
    out = []
    for value, symbol in values:
        while num >= value:
            out.append(symbol)
            num -= value
    return "".join(out)

print("Chỉ 7 ký hiệu:", greedy_seven_symbols(4), greedy_seven_symbols(1994))

Chỉ 7 ký hiệu: IIII MDCCCCLXXXXIIII


# 6. Lời giải chính

```python
VALUES = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
          (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
          (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]

for value, symbol in VALUES:
    while num >= value:
        viết symbol, num -= value
```

Có thể thay `while` bằng `divmod`: `count, num = divmod(num, value)` rồi viết `symbol * count`.

In [10]:
VALUES = [
    (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
    (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
    (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
]

def int_to_roman_greedy(num: int) -> str:
    out = []
    for value, symbol in VALUES:
        count, num = divmod(num, value)
        out.append(symbol * count)
    return "".join(out)

for n in [3749, 58, 1994, 4, 3999]:
    print(n, "->", int_to_roman_greedy(n))

3749 -> MMMDCCXLIX
58 -> LVIII
1994 -> MCMXCIV
4 -> IV
3999 -> MMMCMXCIX


`symbol * count` lặp chuỗi: `"M" * 3 = "MMM"`, `"CM" * 0 = ""`.

# 7. Mô phỏng từng bước: `num = 1994`

| value | symbol | num trước | count = num // value | viết | num sau |
|---|---|---|---|---|---|
| 1000 | M | 1994 | 1 | M | 994 |
| 900 | CM | 994 | 1 | CM | 94 |
| 500 | D | 94 | 0 | | 94 |
| 400 | CD | 94 | 0 | | 94 |
| 100 | C | 94 | 0 | | 94 |
| 90 | XC | 94 | 1 | XC | 4 |
| 50…5 | | 4 | 0 | | 4 |
| 4 | IV | 4 | 1 | IV | 0 |
| 1 | I | 0 | 0 | | 0 |

Kết quả: `M + CM + XC + IV = MCMXCIV`.

In [11]:
def trace_int_to_roman(num: int) -> str:
    out = []
    for value, symbol in VALUES:
        count, rest = divmod(num, value)
        if count:
            print(f"value={value:>4} ({symbol:>2}): {num:>4} // {value} = {count} -> viết {symbol * count!r:6} còn {rest}")
            out.append(symbol * count)
        num = rest
    result = "".join(out)
    print("Kết quả:", result)
    return result

trace_int_to_roman(1994)
print()
trace_int_to_roman(3749)

value=1000 ( M): 1994 // 1000 = 1 -> viết 'M'    còn 994
value= 900 (CM):  994 // 900 = 1 -> viết 'CM'   còn 94
value=  90 (XC):   94 // 90 = 1 -> viết 'XC'   còn 4
value=   4 (IV):    4 // 4 = 1 -> viết 'IV'   còn 0
Kết quả: MCMXCIV

value=1000 ( M): 3749 // 1000 = 3 -> viết 'MMM'  còn 749
value= 500 ( D):  749 // 500 = 1 -> viết 'D'    còn 249
value= 100 ( C):  249 // 100 = 2 -> viết 'CC'   còn 49
value=  40 (XL):   49 // 40 = 1 -> viết 'XL'   còn 9
value=   9 (IX):    9 // 9 = 1 -> viết 'IX'   còn 0
Kết quả: MMMDCCXLIX


'MMMDCCXLIX'

# 8. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def intToRoman(self, num: int) -> str:
        values = [
            (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
            (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
            (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
        ]
        out = []
        for value, symbol in values:
            count, num = divmod(num, value)
            out.append(symbol * count)
        return "".join(out)

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: 3749", sol.intToRoman(3749), "MMMDCCXLIX")
check("Ví dụ 2: 58", sol.intToRoman(58), "LVIII")
check("Ví dụ 3: 1994", sol.intToRoman(1994), "MCMXCIV")
check("Nhỏ nhất: 1", sol.intToRoman(1), "I")
check("Lớn nhất: 3999", sol.intToRoman(3999), "MMMCMXCIX")
check("4", sol.intToRoman(4), "IV")
check("9", sol.intToRoman(9), "IX")
check("40", sol.intToRoman(40), "XL")
check("400", sol.intToRoman(400), "CD")
check("444", sol.intToRoman(444), "CDXLIV")
check("2024", sol.intToRoman(2024), "MMXXIV")

Ví dụ 1: 3749        PASS  MMMDCCXLIX
Ví dụ 2: 58          PASS  LVIII
Ví dụ 3: 1994        PASS  MCMXCIV
Nhỏ nhất: 1          PASS  I
Lớn nhất: 3999       PASS  MMMCMXCIX
4                    PASS  IV
9                    PASS  IX
40                   PASS  XL
400                  PASS  CD
444                  PASS  CDXLIV
2024                 PASS  MMXXIV


## Kiểm tra toàn bộ: so hai cách và đi vòng

Vì miền input chỉ có 3999 số, ta kiểm tra **tất cả**:

1. Greedy khớp cách tra bảng.
2. Đọc ngược bằng hàm `roman_to_int` (bài #13) cho lại đúng số ban đầu.
3. Không có ký hiệu nào lặp quá 3 lần.

In [14]:
SYMBOL_VALUE = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

def roman_to_int(s: str) -> int:
    total = 0
    for i in range(len(s)):
        v = SYMBOL_VALUE[s[i]]
        if i + 1 < len(s) and v < SYMBOL_VALUE[s[i + 1]]:
            total -= v
        else:
            total += v
    return total

for n in range(1, 4000):
    r = sol.intToRoman(n)
    assert r == int_to_roman_table(n), n
    assert roman_to_int(r) == n, n
    assert not any(c * 4 in r for c in "IXCM"), r
    assert not any(c * 2 in r for c in "VLD"), r

print("Đã kiểm tra cả 3999 số: đúng, đi vòng chính xác, không lặp sai luật")

Đã kiểm tra cả 3999 số: đúng, đi vòng chính xác, không lặp sai luật


# 9. Vì sao greedy đúng?

Greedy **không phải lúc nào cũng đúng** (ví dụ thối tiền với các tờ `1, 3, 4` cho số `6`: greedy lấy `4+1+1`, tối ưu là `3+3`). Ở đây nó đúng vì cấu trúc đặc biệt của bảng:

1. Bảng chia thành 4 "tầng" theo hàng: `{M}`, `{CM, D, CD, C}`, `{XC, L, XL, X}`, `{IX, V, IV, I}`.
2. Khi xử lý tới tầng trăm, `num < 1000`; mọi giá trị ở tầng trăm đều là bội của 100 và tầng dưới cộng lại tối đa `99`. Vì vậy greedy ở tầng trăm **dùng đúng chữ số hàng trăm** `h = num // 100`, và để lại `num % 100` cho tầng dưới.
3. Trong một tầng, với chữ số `h`, greedy sinh ra:
   - `h = 9` → `CM`; `h = 4` → `CD`;
   - `5 ≤ h ≤ 8` → `D` + `C` × `(h - 5)`;
   - `1 ≤ h ≤ 3` → `C` × `h`.

   Đây đúng là hàng "Trăm" trong bảng ở mục 4, tức đúng luật La Mã.

Nói cách khác: greedy với 13 giá trị **tự động sinh ra** bảng tra mà cách 1 phải gõ tay.

# 10. Độ phức tạp

Bảng có 13 phần tử cố định, và độ dài kết quả tối đa là 15 ký tự (`MMMDCCCLXXXVIII` = 3888).

```text
Time:  O(1)   — 13 bước cố định
Space: O(1)   — chuỗi kết quả tối đa 15 ký tự
```

Nếu muốn tổng quát hóa cho số lớn tùy ý (thêm ký hiệu mới), thời gian là `O(số ký hiệu + độ dài kết quả)`.

# 11. Những lỗi dễ mắc

## Lỗi 1: Thiếu dạng trừ trong bảng

Chỉ dùng 7 ký hiệu cho ra `IIII`, `DCCCC`… (đã thấy ở mục 5).

## Lỗi 2: Bảng không xếp giảm dần

Greedy **bắt buộc** xét giá trị lớn trước. Nếu đặt `(400, "CD")` trước `(500, "D")`, số `500` sẽ thành `CDC`:

In [15]:
def wrong_order(num: int) -> str:
    values = [(1000, "M"), (900, "CM"), (400, "CD"), (500, "D"),   # SAI thứ tự
              (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
              (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
    out = []
    for value, symbol in values:
        count, num = divmod(num, value)
        out.append(symbol * count)
    return "".join(out)

print("Sai :", wrong_order(500), "| Đúng:", int_to_roman_greedy(500))
print("Sai :", wrong_order(800), "| Đúng:", int_to_roman_greedy(800))

Sai : CDC | Đúng: D
Sai : CDCD | Đúng: DCCC


## Lỗi 3: Dùng dict thay vì list mà không chắc thứ tự

Từ Python 3.7, dict giữ thứ tự chèn, nên dict `{1000: "M", 900: "CM", ...}` vẫn chạy đúng **nếu** bạn chèn đúng thứ tự giảm dần. Nhưng dùng list các cặp thể hiện ý đồ "thứ tự quan trọng" rõ ràng hơn.

## Lỗi 4: Tự chế dạng trừ không hợp lệ

`99` **không** phải `IC`, `49` không phải `IL`, `990` không phải `XM`. Chỉ có 6 dạng trừ. `99 = XC + IX = XCIX`.

# 12. Mẹo nhận diện pattern

**Greedy với bảng "mệnh giá":**

- Cần biểu diễn một số thành tổng các "đơn vị" có sẵn (tiền, ký hiệu, thời gian giờ/phút/giây…).
- Bảng xếp **giảm dần**, mỗi bước lấy đơn vị lớn nhất còn vừa.
- Luôn tự hỏi: *greedy có đúng với bộ mệnh giá này không?* Nếu không chắc, thử phản ví dụ nhỏ hoặc kiểm tra brute force.

**Mẹo "biến ngoại lệ thành mệnh giá":** thay vì viết `if` cho các trường hợp 4 và 9, ta thêm chúng vào bảng. Code ngắn hơn và không còn nhánh đặc biệt.

Bài liên quan: **#13 Roman to Integer**, **#273 Integer to English Words** (Hard), **#322 Coin Change** (nơi greedy **sai**, cần DP).

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Viết số La Mã cho `2468` và `3888` bằng greedy, ghi rõ từng bước (`value`, `count`, phần còn lại).

## Bài 2 — Đoán output

```python
print("XC" * 2)
print(divmod(1994, 1000))
print(int_to_roman_greedy(3000) + int_to_roman_greedy(888))
print(len(int_to_roman_greedy(3888)))
```

## Bài 3 — Điền chỗ trống

In [16]:
def exercise_3_int_to_roman(num: int) -> str:
    values = [
        # TODO 1: điền đủ 13 cặp (giá trị, ký hiệu), xếp GIẢM DẦN
    ]
    out = []
    # TODO 2: với mỗi cặp, lấy count = num // value, viết symbol * count, cập nhật num
    return "".join(out)


# assert exercise_3_int_to_roman(1994) == "MCMXCIV"
# assert exercise_3_int_to_roman(58) == "LVIII"
# assert exercise_3_int_to_roman(3999) == "MMMCMXCIX"

## Bài 4 — Biến thể: thời gian dạng chữ

Đổi một số giây thành chuỗi dạng `"Xh Ym Zs"`, bỏ qua các đơn vị bằng 0. Đây cũng là greedy với "mệnh giá" `3600, 60, 1`.

```python
format_duration(3725) -> "1h 2m 5s"
format_duration(60)   -> "1m"
format_duration(0)    -> "0s"
```

In [17]:
def exercise_4_format_duration(seconds: int) -> str:
    # TODO
    pass


# assert exercise_4_format_duration(3725) == "1h 2m 5s"
# assert exercise_4_format_duration(60) == "1m"
# assert exercise_4_format_duration(0) == "0s"
# assert exercise_4_format_duration(7201) == "2h 1s"

## Bài 5 — Mini challenge: số La Mã ngắn nhất và dài nhất

Trong các số từ `1` tới `3999`:

1. Số nào có biểu diễn La Mã **dài nhất**? Dài bao nhiêu ký tự?
2. Có bao nhiêu số mà biểu diễn chỉ dài **1 ký tự**?

Viết hàm trả về `(số_dài_nhất, độ_dài, số_lượng_số_1_ký_tự)`.

In [18]:
def exercise_5_roman_length_stats():
    # TODO
    pass


# assert exercise_5_roman_length_stats() == (3888, 15, 7)

# 14. Gợi ý

**Bài 1**
- Mức 1: `2468 = 2000 + 400 + 60 + 8`.
- Mức 2: `400 = CD`, `60 = LX`, `8 = VIII`.
- Mức 3: `3888` dùng toàn ký hiệu "thường", không có dạng trừ.

**Bài 3**
- Mức 1: các cặp trừ nằm ngay trước ký hiệu "5" tương ứng: `900` trước `500`, `400` sau `500`.
- Mức 2: thứ tự là `1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1`.
- Mức 3: `count, num = divmod(num, value)`.

**Bài 4**
- Mức 1: mệnh giá `[(3600, "h"), (60, "m"), (1, "s")]`.
- Mức 2: chỉ thêm phần tử khi `count > 0`.
- Mức 3: nếu danh sách rỗng (seconds = 0) thì trả `"0s"`.

**Bài 5**
- Mức 1: chỉ có 3999 số — duyệt hết được.
- Mức 2: dùng `max` với `key=lambda n: len(...)`.
- Mức 3: số có 1 ký tự chính là giá trị của 7 ký hiệu gốc.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
trace_int_to_roman(2468)
print()
trace_int_to_roman(3888)

value=1000 ( M): 2468 // 1000 = 2 -> viết 'MM'   còn 468
value= 400 (CD):  468 // 400 = 1 -> viết 'CD'   còn 68
value=  50 ( L):   68 // 50 = 1 -> viết 'L'    còn 18
value=  10 ( X):   18 // 10 = 1 -> viết 'X'    còn 8
value=   5 ( V):    8 // 5 = 1 -> viết 'V'    còn 3
value=   1 ( I):    3 // 1 = 3 -> viết 'III'  còn 0
Kết quả: MMCDLXVIII

value=1000 ( M): 3888 // 1000 = 3 -> viết 'MMM'  còn 888
value= 500 ( D):  888 // 500 = 1 -> viết 'D'    còn 388
value= 100 ( C):  388 // 100 = 3 -> viết 'CCC'  còn 88
value=  50 ( L):   88 // 50 = 1 -> viết 'L'    còn 38
value=  10 ( X):   38 // 10 = 3 -> viết 'XXX'  còn 8
value=   5 ( V):    8 // 5 = 1 -> viết 'V'    còn 3
value=   1 ( I):    3 // 1 = 3 -> viết 'III'  còn 0
Kết quả: MMMDCCCLXXXVIII


'MMMDCCCLXXXVIII'

## Đáp án Bài 2

In [20]:
print("XC" * 2)                                                # XCXC
print(divmod(1994, 1000))                                      # (1, 994)
print(int_to_roman_greedy(3000) + int_to_roman_greedy(888))    # MMM + DCCCLXXXVIII
print(len(int_to_roman_greedy(3888)))                          # 15

XCXC
(1, 994)
MMMDCCCLXXXVIII
15


## Đáp án Bài 3

In [21]:
def answer_3_int_to_roman(num: int) -> str:
    values = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
    ]
    out = []
    for value, symbol in values:
        count, num = divmod(num, value)
        out.append(symbol * count)
    return "".join(out)

assert answer_3_int_to_roman(1994) == "MCMXCIV"
assert answer_3_int_to_roman(58) == "LVIII"
assert answer_3_int_to_roman(3999) == "MMMCMXCIX"
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [22]:
def answer_4_format_duration(seconds: int) -> str:
    parts = []
    for value, unit in [(3600, "h"), (60, "m"), (1, "s")]:
        count, seconds = divmod(seconds, value)
        if count:
            parts.append(f"{count}{unit}")
    return " ".join(parts) if parts else "0s"

assert answer_4_format_duration(3725) == "1h 2m 5s"
assert answer_4_format_duration(60) == "1m"
assert answer_4_format_duration(0) == "0s"
assert answer_4_format_duration(7201) == "2h 1s"
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [23]:
def answer_5_roman_length_stats():
    longest = max(range(1, 4000), key=lambda n: len(int_to_roman_greedy(n)))
    one_char = sum(1 for n in range(1, 4000) if len(int_to_roman_greedy(n)) == 1)
    return longest, len(int_to_roman_greedy(longest)), one_char

print(answer_5_roman_length_stats(), int_to_roman_greedy(3888))
assert answer_5_roman_length_stats() == (3888, 15, 7)
print("Bài 5 PASS")

(3888, 15, 7) MMMDCCCLXXXVIII


Bài 5 PASS

# 16. Tổng kết

> Đưa 6 dạng trừ vào bảng mệnh giá, xếp giảm dần, rồi greedy: mỗi bước lấy ký hiệu lớn nhất còn vừa.

## Công thức nhớ nhanh

```python
for value, symbol in [(1000,"M"),(900,"CM"),(500,"D"),(400,"CD"),
                      (100,"C"),(90,"XC"),(50,"L"),(40,"XL"),
                      (10,"X"),(9,"IX"),(5,"V"),(4,"IV"),(1,"I")]:
    count, num = divmod(num, value)
    out.append(symbol * count)
```

---

## Checklist trước khi nộp

- [ ] Bảng có đủ **13** giá trị (gồm 6 dạng trừ)?
- [ ] Bảng xếp **giảm dần**?
- [ ] Đã thử 1, 4, 9, 40, 90, 400, 900, 3999?
- [ ] Đi vòng `roman_to_int(intToRoman(n)) == n`?